In [ ]:
# Install required dependencies
%pip install -q \
    langchain \
    pypdf \
    python-docx \
    markdown \
    beautifulsoup4 \
    boto3 \
    tqdm


In [ ]:
import os
from typing import List, Optional, Dict, Any
from pathlib import Path
from tqdm import tqdm

import boto3
from langchain.document_loaders import (
    PyPDFLoader,
    Docx2txtLoader,
    UnstructuredMarkdownLoader,
    BSHTMLLoader,
    TextLoader
)
from langchain.schema import Document

class MultiSourceDocumentLoader:
    """A document loader that can handle multiple file types from both local and S3 sources."""
    
    SUPPORTED_EXTENSIONS = {
        '.pdf': PyPDFLoader,
        '.docx': Docx2txtLoader,
        '.md': UnstructuredMarkdownLoader,
        '.html': BSHTMLLoader,
        '.txt': TextLoader
    }
    
    def __init__(self, local_dir: Optional[str] = None, s3_bucket: Optional[str] = None, 
                 s3_prefix: Optional[str] = None):
        self.local_dir = Path(local_dir) if local_dir else None
        self.s3_bucket = s3_bucket
        self.s3_prefix = s3_prefix
        self.s3_client = boto3.client('s3') if s3_bucket else None
        
    def _is_supported_file(self, filename: str) -> bool:
        """Check if the file type is supported."""
        return any(filename.lower().endswith(ext) for ext in self.SUPPORTED_EXTENSIONS.keys())
    
    def _get_loader_class(self, filename: str):
        """Get the appropriate loader class for a file."""
        ext = Path(filename).suffix.lower()
        return self.SUPPORTED_EXTENSIONS.get(ext)
    
    def _load_local_file(self, filepath: Path) -> List[Document]:
        """Load a single local file."""
        loader_class = self._get_loader_class(str(filepath))
        if not loader_class:
            return []
        
        try:
            loader = loader_class(str(filepath))
            docs = loader.load()
            # Add source metadata
            for doc in docs:
                doc.metadata.update({
                    'source_type': 'local',
                    'file_path': str(filepath),
                    'file_type': filepath.suffix[1:],  # Remove the dot
                })
            return docs
        except Exception as e:
            print(f"Error loading {filepath}: {str(e)}")
            return []
    
    def _load_s3_file(self, key: str) -> List[Document]:
        """Load a single file from S3."""
        if not self._is_supported_file(key):
            return []
            
        try:
            # Download to temporary file
            tmp_path = f"/tmp/{os.path.basename(key)}"
            self.s3_client.download_file(self.s3_bucket, key, tmp_path)
            
            loader_class = self._get_loader_class(key)
            loader = loader_class(tmp_path)
            docs = loader.load()
            
            # Add source metadata
            for doc in docs:
                doc.metadata.update({
                    'source_type': 's3',
                    'bucket': self.s3_bucket,
                    'key': key,
                    'file_type': Path(key).suffix[1:],  # Remove the dot
                })
            
            # Cleanup
            os.remove(tmp_path)
            return docs
        except Exception as e:
            print(f"Error loading {key} from S3: {str(e)}")
            return []
    
    def load(self) -> List[Document]:
        """Load documents from all configured sources."""
        documents = []
        
        # Load local files
        if self.local_dir:
            print("Loading local documents...")
            for filepath in tqdm(list(self.local_dir.rglob('*'))):
                if filepath.is_file() and self._is_supported_file(str(filepath)):
                    documents.extend(self._load_local_file(filepath))
        
        # Load S3 files
        if self.s3_bucket:
            print("Loading S3 documents...")
            paginator = self.s3_client.get_paginator('list_objects_v2')
            prefix = self.s3_prefix or ''
            
            for page in paginator.paginate(Bucket=self.s3_bucket, Prefix=prefix):
                if 'Contents' not in page:
                    continue
                    
                for obj in tqdm(page['Contents']):
                    key = obj['Key']
                    if self._is_supported_file(key):
                        documents.extend(self._load_s3_file(key))
        
        return documents


In [ ]:
# Example 1: Loading from local directory
local_docs_path = "data/documents"  # Update this path to your local documents folder

# Create loader for local files only
local_loader = MultiSourceDocumentLoader(local_dir=local_docs_path)

# Load documents
local_documents = local_loader.load()

print(f"Loaded {len(local_documents)} documents from local directory")
for doc in local_documents[:3]:  # Show first 3 documents
    print(f"\nDocument from {doc.metadata['file_path']}")
    print(f"Content preview: {doc.page_content[:100]}...")


In [ ]:
# Example 2: Loading from S3 bucket
s3_bucket = "your-bucket-name"  # Update with your S3 bucket name
s3_prefix = "documents/"        # Optional: specify a prefix to load from a specific folder

# Create loader for S3 files only
s3_loader = MultiSourceDocumentLoader(s3_bucket=s3_bucket, s3_prefix=s3_prefix)

# Load documents
s3_documents = s3_loader.load()

print(f"Loaded {len(s3_documents)} documents from S3")
for doc in s3_documents[:3]:  # Show first 3 documents
    print(f"\nDocument from s3://{doc.metadata['bucket']}/{doc.metadata['key']}")
    print(f"Content preview: {doc.page_content[:100]}...")


In [ ]:
# Example 3: Loading from both local and S3 sources
combined_loader = MultiSourceDocumentLoader(
    local_dir=local_docs_path,
    s3_bucket=s3_bucket,
    s3_prefix=s3_prefix
)

# Load documents from both sources
all_documents = combined_loader.load()

print(f"Loaded {len(all_documents)} documents in total")
print(f"Documents by source type:")
source_counts = {}
for doc in all_documents:
    source_type = doc.metadata['source_type']
    source_counts[source_type] = source_counts.get(source_type, 0) + 1

for source_type, count in source_counts.items():
    print(f"- {source_type}: {count} documents")


In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

# Initialize the text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len,
    is_separator_regex=False,
)

# Split documents into chunks
split_docs = []
for doc in all_documents:
    # Split the document
    doc_chunks = text_splitter.split_text(doc.page_content)
    
    # Create new Document objects for each chunk
    for i, chunk in enumerate(doc_chunks):
        # Copy the original metadata
        chunk_metadata = doc.metadata.copy()
        # Add chunk-specific metadata
        chunk_metadata.update({
            'chunk_index': i,
            'total_chunks': len(doc_chunks),
            'chunk_size': len(chunk)
        })
        
        split_docs.append(Document(
            page_content=chunk,
            metadata=chunk_metadata
        ))

print(f"Created {len(split_docs)} chunks from {len(all_documents)} documents")
print("\nExample chunk metadata:")
if split_docs:
    print(split_docs[0].metadata)
